# EduHinglish — Phase 6: IndicBART Hinglish Generator Fine-tuning

This notebook fine-tunes **ai4bharat/IndicBART** (MBartForConditionalGeneration, 244M params)
with **LoRA/PEFT** to generate natural Hinglish explanations from NCERT English text.

## What this notebook does
1. Installs required packages
2. Mounts Google Drive and loads the unified dataset
3. Prepares seq2seq (source, target) pairs — Types A, B, C
4. Fine-tunes IndicBART with LoRA adapters using `Seq2SeqTrainer`
5. Tests generation with sample NCERT sentences
6. Saves the fine-tuned model to Google Drive

## Expected runtime
- **~30–45 minutes** on a free Colab T4 GPU (5 epochs, ~2300 pairs)
- Runs significantly slower on CPU — use GPU runtime

> **Before running:** Go to **Runtime → Change runtime type** and select **GPU (T4)**.

In [ ]:
# Cell 2 — GPU Check + Install Dependencies
import os, sys

IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IS_COLAB:
    os.system('nvidia-smi')
    os.system(
        'pip install -q '
        'transformers>=4.40.0 '
        'datasets>=2.19.0 '
        'peft>=0.10.0 '
        'accelerate>=0.29.3 '
        'sentencepiece>=0.2.0 '
        'colorama>=0.4.6 '
        'sentence-transformers>=2.7.0'
    )
    print('Packages installed. IMPORTANT: Go to Runtime → Restart session, then re-run all cells.')
else:
    print('Running locally — skipping pip install and nvidia-smi')

import torch
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('[WARNING] No GPU detected — training will be very slow.')

In [ ]:
# Cell 3 — Mount Google Drive + Clone / Upload Dataset
from google.colab import drive
drive.mount('/content/drive')

# Option A: Clone repo from GitHub (recommended)
# !git clone https://github.com/PersuasivePost/EduHinglish.git
# %cd EduHinglish

# Option B: Dataset already on Drive — set path below
import os
DATASET_PATH = '/content/drive/MyDrive/EduHinglish/data/unified_biology_dataset.json'
SEQ2SEQ_OUTPUT_DIR = '/content/indicbart_seq2seq_dataset'
MODEL_OUTPUT_DIR   = '/content/indicbart_v1'

print(f'Dataset path : {DATASET_PATH}')
print(f'Seq2seq data : {SEQ2SEQ_OUTPUT_DIR}')
print(f'Model output : {MODEL_OUTPUT_DIR}')

In [ ]:
# Cell 4 — Verify Dataset
import json

with open(DATASET_PATH, encoding='utf-8') as f:
    data = json.load(f)

print(f'Dataset : {len(data)} entries')
print(f'Sample  : {data[0]["original_english"][:80]}...')
print(f'Target  : {data[0]["hinglish_roman"][:80]}...')
print()

# Quick schema check
required_keys = ['id', 'original_english', 'hinglish_roman', 'topic', 'chapter']
missing = [k for k in required_keys if k not in data[0]]
if missing:
    print(f'[WARNING] Missing keys in first entry: {missing}')
else:
    print('[OK] Dataset schema looks good.')

In [ ]:
# Cell 5 — Step 1: Prepare Seq2Seq Data (Types A, B, C)
import json
import random
from pathlib import Path
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

MODEL_NAME        = 'ai4bharat/IndicBART'
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256
MIN_WORDS         = 5
SEED              = 42

# ── Filter ──────────────────────────────────────────────────────────────────
def has_enough_words(text, n=MIN_WORDS):
    return isinstance(text, str) and len(text.split()) >= n

entries = [e for e in data
           if has_enough_words(e.get('original_english', ''))
           and has_enough_words(e.get('hinglish_roman', ''))]
print(f'Filtered entries : {len(entries)} / {len(data)}')

# ── Build pairs ─────────────────────────────────────────────────────────────
pairs = []

# Type A — Direct Translation
for e in entries:
    pairs.append({'source': f"Translate to Hinglish: {e['original_english']}",
                  'target': e['hinglish_roman']})
type_a_count = len(entries)

# Type B — Query-Grounded (student query entries only)
type_b_start = len(pairs)
for e in entries:
    if e.get('is_student_query'):
        pairs.append({
            'source': f"Explain in Hinglish: {e['original_english']} | Student asks: {e['hinglish_roman']}",
            'target': e['hinglish_roman']
        })
type_b_count = len(pairs) - type_b_start

# Type C — Topic-Conditioned
for e in entries:
    topic = (e.get('topic') or 'General').strip()
    pairs.append({'source': f"Topic: {topic} | Explain in Hinglish: {e['original_english']}",
                  'target': e['hinglish_roman']})
type_c_count = len(entries)

print(f'Type A pairs : {type_a_count}')
print(f'Type B pairs : {type_b_count}')
print(f'Type C pairs : {type_c_count}')
print(f'Total pairs  : {len(pairs)}')

# ── Tokenise ────────────────────────────────────────────────────────────────
print(f'\nLoading tokenizer: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'[OK] Tokenizer loaded (vocab size: {tokenizer.vocab_size})')

print(f'Tokenizing {len(pairs)} pairs ...')
examples = []
for p in pairs:
    src = tokenizer(p['source'], max_length=MAX_SOURCE_LENGTH, truncation=True, padding=False)
    tgt = tokenizer(p['target'], max_length=MAX_TARGET_LENGTH, truncation=True, padding=False)
    examples.append({
        'input_ids':      src['input_ids'],
        'attention_mask': src['attention_mask'],
        'labels':         tgt['input_ids'],
    })
print(f'Tokenization complete.')

# ── Split 80/20 ──────────────────────────────────────────────────────────────
random.seed(SEED)
random.shuffle(examples)
cut = max(1, int(len(examples) * 0.8))
train_examples = examples[:cut]
dev_examples   = examples[cut:]

# ── Save DatasetDict ─────────────────────────────────────────────────────────
def to_dataset(exs):
    return Dataset.from_dict({
        'input_ids':      [e['input_ids']      for e in exs],
        'attention_mask': [e['attention_mask']  for e in exs],
        'labels':         [e['labels']          for e in exs],
    })

ds = DatasetDict({'train': to_dataset(train_examples), 'dev': to_dataset(dev_examples)})
Path(SEQ2SEQ_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
ds.save_to_disk(SEQ2SEQ_OUTPUT_DIR)

# Save config
config = {
    'model_name': MODEL_NAME,
    'max_source_length': MAX_SOURCE_LENGTH,
    'max_target_length': MAX_TARGET_LENGTH,
    'total_pairs':  len(pairs),
    'type_a_count': type_a_count,
    'type_b_count': type_b_count,
    'type_c_count': type_c_count,
    'train_count':  len(train_examples),
    'dev_count':    len(dev_examples),
}
with open(Path(SEQ2SEQ_OUTPUT_DIR) / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)

avg_src = sum(len(e['input_ids']) for e in examples) / len(examples)
avg_tgt = sum(len(e['labels'])    for e in examples) / len(examples)
print(f'\n[OK] Dataset saved to {SEQ2SEQ_OUTPUT_DIR}/')
print(f'     Train: {len(train_examples)}  Dev: {len(dev_examples)}')
print(f'     Avg source tokens: {avg_src:.1f}  Avg target tokens: {avg_tgt:.1f}')

# Print 3 sample pairs
print('\n--- Sample pairs ---')
for i, ex in enumerate(examples[:3]):
    src_text = tokenizer.decode(ex['input_ids'],  skip_special_tokens=True)
    tgt_text = tokenizer.decode(ex['labels'],     skip_special_tokens=True)
    print(f'\nExample {i+1}')
    print(f'  SRC ({len(ex["input_ids"])} tok): {src_text[:110]}')
    print(f'  TGT ({len(ex["labels"])} tok)  : {tgt_text[:110]}')

In [ ]:
# Cell 6 — Step 2: Fine-tune IndicBART with LoRA
import time
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import torch
from transformers import (
    MBartForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model
from datasets import load_from_disk

# ── Config ──────────────────────────────────────────────────────────────────
EPOCHS     = 5
BATCH_SIZE = 8      # reduce to 4 if OOM
LR         = 3e-4
LORA_R     = 16
LORA_ALPHA = 32
USE_FP16   = torch.cuda.is_available()

# ── Load dataset ─────────────────────────────────────────────────────────────
ds       = load_from_disk(SEQ2SEQ_OUTPUT_DIR)
train_ds = ds['train']
dev_ds   = ds['dev']
print(f'Train: {len(train_ds)}  Dev: {len(dev_ds)}')

# ── Load model ───────────────────────────────────────────────────────────────
print(f'\nLoading {MODEL_NAME} ...')
model = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.1,
    bias='none',
    target_modules=['q_proj', 'v_proj'],
)
model = get_peft_model(model, lora_cfg)
trainable, total = model.get_nb_trainable_parameters()
print(f'Trainable params: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)')

# ── Custom data collator ─────────────────────────────────────────────────────
@dataclass
class Seq2SeqDataCollator:
    pad_token_id: int
    label_pad_id: int = -100

    def __call__(self, features: list[dict[str, Any]]) -> dict[str, Any]:
        max_src = max(len(f['input_ids']) for f in features)
        max_tgt = max(len(f['labels'])    for f in features)
        input_ids_batch, attn_mask_batch, labels_batch = [], [], []
        for feat in features:
            src_pad = max_src - len(feat['input_ids'])
            tgt_pad = max_tgt - len(feat['labels'])
            input_ids_batch.append(feat['input_ids']      + [self.pad_token_id] * src_pad)
            attn_mask_batch.append(feat['attention_mask'] + [0]                 * src_pad)
            labels_batch.append(   feat['labels']         + [self.label_pad_id] * tgt_pad)
        return {
            'input_ids':      torch.tensor(input_ids_batch, dtype=torch.long),
            'attention_mask': torch.tensor(attn_mask_batch, dtype=torch.long),
            'labels':         torch.tensor(labels_batch,    dtype=torch.long),
        }

pad_id       = tokenizer.pad_token_id or 0
data_collator = Seq2SeqDataCollator(pad_token_id=pad_id)

# ── Training arguments ───────────────────────────────────────────────────────
Path(MODEL_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=USE_FP16,
    predict_with_generate=True,
    generation_max_length=256,
    logging_steps=50,
    report_to='none',
    seed=42,
    remove_unused_columns=False,
)

# ── Train ────────────────────────────────────────────────────────────────────
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=data_collator,
)

print(f'\nStarting training: {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LR}, fp16={USE_FP16}')
start = time.time()
train_result = trainer.train()
elapsed = time.time() - start
fmt_time = lambda s: f'{s:.0f}s' if s < 60 else (f'{s/60:.1f} min' if s < 3600 else f'{s/3600:.1f} h')
print(f'\nTraining complete in {fmt_time(elapsed)}')

# Extract final losses
log_history = trainer.state.log_history
final_train_loss = next((e['loss']      for e in reversed(log_history) if 'loss'      in e), None)
final_eval_loss  = next((e['eval_loss'] for e in reversed(log_history) if 'eval_loss' in e), None)
best_ckpt = getattr(trainer.state, 'best_model_checkpoint', None) or MODEL_OUTPUT_DIR

# ── Save model + summary ─────────────────────────────────────────────────────
model.save_pretrained(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

summary = {
    'model_name':           MODEL_NAME,
    'epochs':               EPOCHS,
    'final_train_loss':     round(final_train_loss, 4) if final_train_loss else None,
    'final_eval_loss':      round(final_eval_loss,  4) if final_eval_loss  else None,
    'total_training_time':  fmt_time(elapsed),
    'training_time_seconds': round(elapsed, 1),
    'trainable_params':     trainable,
    'total_params':         total,
    'lora_r':               LORA_R,
    'lora_alpha':           LORA_ALPHA,
    'batch_size':           BATCH_SIZE,
    'learning_rate':        LR,
    'best_checkpoint_path': best_ckpt,
    'device':               'cuda' if USE_FP16 else 'cpu',
    'fp16':                 USE_FP16,
}
with open(Path(MODEL_OUTPUT_DIR) / 'training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'[OK] LoRA adapter saved : {MODEL_OUTPUT_DIR}/')
if final_train_loss: print(f'     Final train loss : {final_train_loss:.4f}')
if final_eval_loss:  print(f'     Final eval  loss : {final_eval_loss:.4f}')

In [ ]:
# Cell 7 — Plot Training Curves
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_loss   = [e['loss']      for e in log_history if 'loss'      in e]
train_epochs = [e['epoch']     for e in log_history if 'loss'      in e]
eval_loss    = [e['eval_loss'] for e in log_history if 'eval_loss' in e]
eval_epochs  = [e['epoch']     for e in log_history if 'eval_loss' in e]

max_epoch   = int(max(eval_epochs)) if eval_epochs else EPOCHS
epoch_ticks = range(1, max_epoch + 1)

plt.figure(figsize=(10, 4))
plt.plot(train_epochs, train_loss, label='Training Loss',   color='#1f77b4', linewidth=1.5)
if eval_loss:
    plt.plot(eval_epochs, eval_loss, label='Validation Loss', color='#ff7f0e', marker='o', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('IndicBART Fine-tuning: Training & Validation Loss')
plt.xticks(epoch_ticks)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8 — Step 3: Interactive Generation Test
# Loads the fine-tuned model from MODEL_OUTPUT_DIR and generates Hinglish
# for 5 NCERT-style English sentences.

from transformers import AutoTokenizer, MBartForConditionalGeneration
from peft import PeftModel
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load base model + LoRA adapter
print(f'Loading fine-tuned model from {MODEL_OUTPUT_DIR} ...')
_tokenizer = AutoTokenizer.from_pretrained(MODEL_OUTPUT_DIR)
_base       = MBartForConditionalGeneration.from_pretrained('ai4bharat/IndicBART')
_gen_model  = PeftModel.from_pretrained(_base, MODEL_OUTPUT_DIR).to(device)
_gen_model.eval()
print('[OK] Model loaded.')

def generate_hinglish(english_text: str, prefix: str = 'Translate to Hinglish: ') -> str:
    """Generate a Hinglish explanation from English input."""
    src = _tokenizer(
        prefix + english_text,
        max_length=256,
        truncation=True,
        return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        out = _gen_model.generate(
            input_ids=src['input_ids'],
            attention_mask=src['attention_mask'],
            max_length=256,
            num_beams=4,
            early_stopping=True,
        )
    return _tokenizer.decode(out[0], skip_special_tokens=True)

# Test with 5 NCERT-style sentences
test_sentences = [
    'Photosynthesis is the process by which green plants make food using sunlight.',
    'The nucleus contains chromosomes which carry genetic information.',
    'Respiration involves breaking down of glucose to release energy.',
    'Stomata are tiny pores on leaves that allow gas exchange.',
    'DNA replication is the process of making an identical copy of DNA.',
]

print('\n' + '='*60)
print('  Generation Test — 5 NCERT Sample Sentences')
print('='*60)
for i, eng in enumerate(test_sentences, 1):
    hin = generate_hinglish(eng)
    print(f'\nExample {i}')
    print(f'  ENG : {eng}')
    print(f'  HIN : {hin}')
print('\n' + '='*60)

In [ ]:
# Cell 9 — Step 4: Save Model to Google Drive
import shutil, os
from pathlib import Path

DRIVE_SAVE_DIR = '/content/drive/MyDrive/EduHinglish_models/indicbart_v1'
Path(DRIVE_SAVE_DIR).mkdir(parents=True, exist_ok=True)

print(f'Copying {MODEL_OUTPUT_DIR} → {DRIVE_SAVE_DIR} ...')
shutil.copytree(MODEL_OUTPUT_DIR, DRIVE_SAVE_DIR, dirs_exist_ok=True)
print('[OK] Model saved to Google Drive.')
print(f'     Path: {DRIVE_SAVE_DIR}')
print()

# List saved files
saved = list(Path(DRIVE_SAVE_DIR).iterdir())
print('Saved files:')
for f in sorted(saved):
    size = f.stat().st_size if f.is_file() else 0
    size_str = f'{size/1e6:.1f} MB' if size > 1e6 else f'{size/1e3:.1f} KB'
    print(f'  {f.name:40s}  {size_str if f.is_file() else "[dir]"}')

## Results Summary

| Metric | Value |
|--------|-------|
| Model | IndicBART (ai4bharat/IndicBART) + LoRA |
| LoRA rank / alpha | 16 / 32 |
| Target modules | q_proj, v_proj |
| Training epochs | 5 |
| Batch size | 8 (grad accum ×2 → effective 16) |
| Learning rate | 3e-4 |
| Pair types | A (direct) + B (query-grounded) + C (topic-conditioned) |

### Sample Outputs
*(Fill in after running Cell 8)*

| English Input | Generated Hinglish |
|---------------|--------------------|
| Photosynthesis is the process... | *(generated output)* |
| The nucleus contains chromosomes... | *(generated output)* |

### Next Steps
- **Jatin** can now load `models/indicbart_v1/` in `src/generator.py` (Prompt 4)
- Run `training/evaluate_generator.py` for BLEU / ROUGE / CMI metrics (Prompt 5)
- Integrate into `src/pipeline.py` for the full M1→M2→M3 flow (Prompt 6)